This script refactors the original grid-search approach for finding optimal
ensemble weights, incorporating Bayesian Optimization with Optuna and prediction
caching for dramatically improved efficiency and scientific robustness.

0. Set Up Environment

In [ ]:
!pip install albumentations torchinfo optuna
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

1. Imports

In [ ]:
print("Importing libraries...")
# Install required packages if needed

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image, UnidentifiedImageError
import gc
import time
from tqdm import tqdm
import torchvision.models as models
from datetime import datetime
import torch.nn.functional as F
import random
import shutil
from torch.cuda.amp import autocast
import cv2
import segmentation_models_pytorch as smp
import timm
from torchinfo import summary
import pandas as pd
import json
import warnings
import albumentations as A
from albumentations.pytorch import ToTensorV2
import zipfile
import math
import optuna
import re
from collections import defaultdict

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# --- Configuration & Setup ---
print("Configuring environment...")
# --- Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Ensemble Configuration ---
N_TOP_MODELS = 8 # Number of top models to include in the ensemble
METADATA_DIR = "METADATA_CHECKPOINTS/DIAGSET/REINHARD" # Folder with _meta.json files
SORT_METRIC = "best_val_auprc_score" # Metric in metadata to rank models

N_OPTUNA_TRIALS = 50  # Number of weight combinations to test. 100-200 is a good starting point.
# Primary metric Optuna will maximize when tuning ensemble weights / thresholds.
# Options: 'mcc', 'dice', 'iou'  (we recommend 'mcc' here)
METRIC_TO_OPTIMIZE = 'mcc'

BATCH_SIZE = 32 # Can potentially use a smaller batch size for ensemble inference if memory is tight
WORKERS = 2

# --- Dataset Source Path ---
DATASET_ZIP_DIR = 'IA_MEDICA_SAMPLES/DIAGSET/REINHARD'

# --- Data Extraction Directory ---
base_data_dir = '/content/dataset' # Extracted fold data goes here

# --- Evaluation Configuration ---
val_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER')
val_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER_MASK')
val_not_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER')
val_not_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER_MASK')


# --- Other Config ---
OUTPUT_DIR = "RESULTS_REPORT_ENSEMBLE/DIAGSET/REINHARD" # Specific output dir

DECODER_DROPOUT = 0


WHERE_WAS_CREATED = '/content/drive/MyDrive/Personal_Drive_Bruno/'
CURRENT_ENV = '/content/drive/MyDrive/'

# Set to True to use a smaller, stratified subset of the data for faster training.
USE_SUBSET = True
# The ratio of the full dataset to use (e.g., 0.10 = 10%).
SUBSET_RATIO = 0.25

PRED_CACHE_DIR = '/content/pred_cache'

SEED = 24

TPR_TARGET = 0.95
THRESHOLD_POLICY = "tpr_target"   # or "maximize_metric"
SECONDARY_METRIC = "tnr"          # or "mcc"


# --- Speed knobs for Optuna ---
USE_OPTUNA_SUBSET = False
OPTUNA_SUBSET_SIZE = 10000  # 5k–20k typical sweet spot
OPTUNA_SUBSET_SEED = SEED

OPTUNA_ENSEMBLE_CHUNK = 256  # chunk over N for ensemble build
THRESH_SCAN_CHUNK = 512      # chunk inside threshold scan (keep RAM low)

# Optional: fewer thresholds for speed during trials
THRESH_NUM_STEPS_TRIALS = 100
THRESH_NUM_STEPS_FINAL  = 200


VAL_HOLDOUT_FRAC = 0.20          # 10–20% typical
VAL_HOLDOUT_SEED = SEED + 123
OPTUNA_SUBSET_POS_FRAC = 0.50    # keep your choice


STATS_SAMPLE_SIZE = None #20000
MEAN = [0.5631743144356475, 0.3779451521216607, 0.6975475908629748]
STD = [0.07002276986060542, 0.05347828888148516, 0.047768733057653855]

In [ ]:
def change_path(path, possible_paths=None):
    if possible_paths is None:
        possible_paths = ['/content/drive/MyDrive/Personal_Drive_Bruno/','/content/drive/MyDrive/']
    for p in possible_paths:
      path=os.path.join(p,path)
      if os.path.exists(path):
        print(f"Path changed succesfully: {path}")
        return path
    raise ValueError(f"The path {path} does not match any known environments.")

In [ ]:
METADATA_DIR = change_path(METADATA_DIR)
DATASET_ZIP_DIR = change_path(DATASET_ZIP_DIR)
OUTPUT_DIR = change_path(OUTPUT_DIR)

In [ ]:
# --- Paths within the extracted fold ---
train_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/CANCER')
train_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/CANCER_MASK')
train_not_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER')
train_not_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER_MASK')
val_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER')
val_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER_MASK')
val_not_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER')
val_not_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER_MASK')

In [ ]:
# --- Reproducibility ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Configuration complete.")

In [ ]:
def count_effective_samples(dataloader):
    n = 0
    for batch in dataloader:
        if batch is None:
            continue
        images, masks = batch
        if images is None or masks is None:
            continue
        # Ensure batch dim
        if images.ndim == 3:
            bsz = 1
        else:
            bsz = images.shape[0]
        n += bsz
    return n

In [ ]:
def compute_segmentation_metrics(pred, target, smooth: float = 1e-6):
    """
    Compute tumor Dice (macro over tumor-containing patches) and MCC over all pixels.

    Args:
        pred:   [B, 1, H, W] or [B, H, W] tensor with 0/1 predictions for the cancer class.
        target: [B, 1, H, W] or [B, H, W] tensor with 0/1 ground-truth labels for the cancer class.

    Returns:
        dice_pos_macro (float): mean Dice over patches that contain >= 1 tumor pixel.
        mcc (float): Matthews Correlation Coefficient over all pixels in the batch.
    """
    # Ensure shape [B, H, W]
    if pred.ndim == 4:
        pred = pred.squeeze(1)
    if target.ndim == 4:
        target = target.squeeze(1)

    pred = pred.int()
    target = target.int()

    # Per-image TP, FP, FN, TN
    tp = ((pred == 1) & (target == 1)).sum(dim=(1, 2))
    fp = ((pred == 1) & (target == 0)).sum(dim=(1, 2))
    fn = ((pred == 0) & (target == 1)).sum(dim=(1, 2))
    tn = ((pred == 0) & (target == 0)).sum(dim=(1, 2))

    # ---- Tumor-only macro Dice (ignore pure-negative patches) ----
    has_tumor = (target.sum(dim=(1, 2)) > 0)
    tp_pos = tp[has_tumor]
    fp_pos = fp[has_tumor]
    fn_pos = fn[has_tumor]

    if tp_pos.numel() > 0:
        dice_pos = (2.0 * tp_pos + smooth) / (2.0 * tp_pos + fp_pos + fn_pos + smooth)
        dice_pos_macro = float(dice_pos.mean())
    else:
        # No tumor patches in this batch
        dice_pos_macro = 0.0

    # ---- MCC over ALL pixels in the batch ----
    total_tp = int(tp.sum())
    total_fp = int(fp.sum())
    total_fn = int(fn.sum())
    total_tn = int(tn.sum())

    tp_fp = total_tp + total_fp
    tp_fn = total_tp + total_fn
    tn_fp = total_tn + total_fp
    tn_fn = total_tn + total_fn

    denom_mcc = (tp_fp * tp_fn * tn_fp * tn_fn) ** 0.5
    if denom_mcc > 0:
        mcc = float((total_tp * total_tn - total_fp * total_fn) / (denom_mcc + 1e-12))
    else:
        mcc = 0.0

    return dice_pos_macro, mcc


def mcc_coefficient(pred, target, smooth: float = 1e-6, debug: bool = False):
    """
    Convenience wrapper: same inputs as dice_coefficient, returns only MCC.
    """
    _, mcc = compute_segmentation_metrics(pred, target, smooth=smooth)
    if debug:
        print(f"[DEBUG] MCC = {mcc:.6f}")
    return mcc


def dice_coefficient(pred, target, smooth: float = 1e-6, debug: bool = False):
    """
    New definition: returns tumor Dice (Dice_pos_macro) ONLY.

    - pred:   [B, 1, H, W] or [B, H, W] (0/1)
    - target: [B, 1, H, W] or [B, H, W] (0/1)
    """
    dice_pos_macro, _ = compute_segmentation_metrics(pred, target, smooth=smooth)
    if debug:
        print(f"[DEBUG] Dice_pos_macro = {dice_pos_macro:.6f}")
    # Clamp into [0, 1] and return Python float
    dice_pos_macro = float(dice_pos_macro)
    return max(0.0, min(1.0, dice_pos_macro))

In [ ]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [ ]:
def load_cached_ground_truths(cached_trues_path: str) -> torch.Tensor:
    """
    Load ground-truth tensor once (CPU) to avoid re-loading in every Optuna trial.
    """
    print(f"Loading cached ground truths from: {cached_trues_path}")
    trues = torch.load(cached_trues_path)
    if trues is None or not torch.is_tensor(trues):
        raise RuntimeError("Cached ground truths are missing or not a tensor.")
    print(f"Cached ground truths loaded. Shape: {tuple(trues.shape)}")
    return trues

In [ ]:
def get_model(architecture, encoder,validation=False):

  if validation:
    aux_params=None
  else:
    aux_params=dict(dropout=DECODER_DROPOUT, classes=2)

  encoder_weights = None if validation else "imagenet"

  if architecture=="SWIN":
    model = smp.Unet(
    encoder_name=encoder,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=2,
    activation=None,
    decoder_attention_type=None,
    aux_params=aux_params)
  elif architecture=="DEEPLABV3PLUS":
    model = smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="INCEPTIONRESNETV2":
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="DPT":
    model = smp.DPT(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        decoder_readout='ignore',
        aux_params=aux_params)
  elif architecture=="UNET++":
    model = smp.UnetPlusPlus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="FPN":
    model = smp.FPN(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="SEGFORMER":
    model = smp.Segformer(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="MANET":
    model = smp.MAnet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="UPERNET":
    model = smp.UPerNet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  else:
    raise ValueError(f"Unknown architecture: {architecture}")

  return model

4. Create Custom Dataset

In [ ]:
class ProstateCancerDataset(Dataset):
    def __init__(
        self,
        cancer_image_dir,
        cancer_mask_dir,
        not_cancer_image_dir,
        not_cancer_mask_dir,
        mean=None,
        std=None,
        compute_stats=False,
        stats_sample_size=None,  # e.g. 10_000 patches to speed up estimation
    ):
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        self.patient_ids = []

        # Combine and store file paths and labels
        self.image_paths = []
        self.mask_paths = []
        self.labels = []

        # Compile the regex once for efficiency
        patient_id_pattern = re.compile(r'PATIENT_(\d+)_')

        # --- Process CANCER images ---
        cancer_images = []
        if os.path.isdir(self.cancer_image_dir):
            cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith('.png')]

        for img_name in cancer_images:
            mask_path = os.path.join(self.cancer_mask_dir, img_name)
            match = patient_id_pattern.search(img_name)
            if os.path.isfile(mask_path) and match:
                self.image_paths.append(os.path.join(self.cancer_image_dir, img_name))
                self.mask_paths.append(mask_path)
                self.labels.append(1)
                self.patient_ids.append(match.group(1))

        # --- Process NOT_CANCER images ---
        not_cancer_images = []
        if os.path.isdir(self.not_cancer_image_dir):
            not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith('.png')]

        for img_name in not_cancer_images:
            mask_path = os.path.join(self.not_cancer_mask_dir, img_name)
            match = patient_id_pattern.search(img_name)
            if os.path.isfile(mask_path) and match:
                self.image_paths.append(os.path.join(self.not_cancer_image_dir, img_name))
                self.mask_paths.append(mask_path)
                self.labels.append(0)
                self.patient_ids.append(match.group(1))

        # ------------------------------------------------------------------
        #  NEW: dataset-specific mean/std (after stain normalization)
        # ------------------------------------------------------------------
        if mean is not None and std is not None:
            # Use externally provided stats (recommended for VAL / TEST)
            self.mean = mean if isinstance(mean, list) else list(mean)
            self.std = std if isinstance(std, list) else list(std)
            print(f"[ProstateCancerDataset] Using provided mean/std: "
                  f"mean={self.mean}, std={self.std}")
        elif compute_stats:
            # Compute stats from this dataset (recommended for TRAIN set)
            print("[ProstateCancerDataset] Computing dataset-specific mean/std...")
            self.mean, self.std = self._compute_dataset_mean_std(
                max_samples=stats_sample_size
            )
            print(f"[ProstateCancerDataset] Computed mean: {self.mean}")
            print(f"[ProstateCancerDataset] Computed std:  {self.std}")
        else:
            # Fallback: ImageNet stats (old behavior, but less ideal scientifically)
            self.mean = [0.485, 0.456, 0.406]
            self.std = [0.229, 0.224, 0.225]
            print("[ProstateCancerDataset] Using ImageNet mean/std "
                  "(no dataset-specific stats requested).")

        # --- Base Transformation (Applied to ALL data) ---
        # NOTE: Albumentations.Normalize expects mean/std in [0,1] scale when
        # max_pixel_value=255.0 (default). We computed them in that scale.
        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR),  # Specify interpolation
            A.Normalize(mean=self.mean, std=self.std),
            ToTensorV2(),  # Handles image scaling & channel order
        ])

    def _compute_dataset_mean_std(self, max_samples=None):
        """
        Compute per-channel mean and std over this dataset's images.

        Args:
            max_samples (int or None): if set, randomly sample up to this many
                images to estimate stats (for speed). If None, use all images.

        Returns:
            mean (list of 3 floats), std (list of 3 floats) in [0,1] scale.
        """
        # If dataset is empty, fall back to ImageNet to avoid crashes
        if len(self.image_paths) == 0:
            print("[ProstateCancerDataset] WARNING: No images found; "
                  "falling back to ImageNet stats.")
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        # Decide which indices to use
        indices = np.arange(len(self.image_paths))
        if max_samples is not None and max_samples < len(indices):
            np.random.shuffle(indices)
            indices = indices[:max_samples]

        n_pixels_total = 0
        channel_sum = np.zeros(3, dtype=np.float64)
        channel_sum_sq = np.zeros(3, dtype=np.float64)

        for idx in indices:
            img_path = self.image_paths[idx]
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img is None:
                continue  # skip broken images

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype(np.float32) / 255.0  # scale to [0,1]

            # Flatten to (N, 3)
            img_flat = img.reshape(-1, 3)

            n_pixels = img_flat.shape[0]
            n_pixels_total += n_pixels

            channel_sum += img_flat.sum(axis=0)
            channel_sum_sq += (img_flat ** 2).sum(axis=0)

        if n_pixels_total == 0:
            print("[ProstateCancerDataset] WARNING: Failed to load any pixels "
                  "while computing stats; falling back to ImageNet stats.")
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        mean = channel_sum / n_pixels_total
        var = (channel_sum_sq / n_pixels_total) - mean ** 2
        std = np.sqrt(np.maximum(var, 1e-12))

        return mean.tolist(), std.tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        try:
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None: raise IOError("cv2.imread failed for image")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for consistency if needed downstream

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None: raise IOError("cv2.imread failed for mask")

        except Exception as e:
            print(f"Error loading image/mask: {img_path} / {mask_path} - {e}")
            # Return None tuple, handled by collate_fn
            return None, None

        # Create Two-Channel Mask (One-Hot Encode) before transform
        mask = mask.astype(np.uint8)
        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0  # Background channel
        two_channel_mask[mask != 0, 1] = 1.0  # Cancer channel

        # Apply the base transformations
        try:
            # Pass mask correctly (shape H, W, C)
            augmented = self.base_transform(image=image, mask=two_channel_mask)
            final_image = augmented['image'] # Shape (C, H, W), FloatTensor, Normalized
            final_mask = augmented['mask']   # Shape (C, H, W), FloatTensor, Values 0.0 or 1.0

            # Ensure mask shape is (2, 224, 224)
            if final_mask.shape[0] != 2:
                 resized_mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
                 two_channel_mask_resized = np.zeros((224, 224, 2), dtype=np.float32)
                 two_channel_mask_resized[resized_mask == 0, 0] = 1.0
                 two_channel_mask_resized[resized_mask != 0, 1] = 1.0
                 final_mask = torch.from_numpy(two_channel_mask_resized).permute(2, 0, 1) # HWC -> CHW

            # Final check on mask shape
            if final_mask.shape != (2, 224, 224):
                 raise ValueError(f"Final mask shape is incorrect: {final_mask.shape}")


        except Exception as e:
             print(f"Error applying transform to {os.path.basename(img_path)}: {e}")
             return None, None # Return None tuple on transform error


        return final_image, final_mask

    def get_class_counts(self):
        """Returns the counts of cancer (1) and not-cancer (0) samples."""
        counts = np.bincount(self.labels)
        not_cancer_count = counts[0] if len(counts) > 0 else 0
        cancer_count = counts[1] if len(counts) > 1 else 0
        return {'CANCER': cancer_count, 'NOT_CANCER': not_cancer_count}

print("Dataset definition complete.")

In [ ]:
def create_stratified_subset(full_dataset, ratio, split_name="Unknown"):
    """
    Creates a scientifically robust, stratified random subsample of a dataset.

    This function performs a two-level stratification:
    1. It groups all patches by their source patient.
    2. Within each patient, it further groups patches by class (Cancer/Not Cancer).
    3. It then samples the specified ratio from each of these sub-groups.

    This ensures the final subset precisely preserves the class proportions
    within each patient from the original full dataset.
    """
    print(f"  Creating a {ratio:.0%} two-level stratified subsample for {split_name.upper()} (by patient and class)...")

    indices_by_patient_and_class = defaultdict(lambda: defaultdict(list))
    for i in range(len(full_dataset)):
        # These attributes must exist on the dataset object
        patient_id = full_dataset.patient_ids[i]
        label = full_dataset.labels[i]
        indices_by_patient_and_class[patient_id][label].append(i)

    print(f"    Found {len(indices_by_patient_and_class)} unique patients in the {split_name} split.")

    subset_indices = []
    generator = torch.Generator().manual_seed(SEED)

    for patient_id, class_groups in indices_by_patient_and_class.items():
        for label, indices in class_groups.items():
            num_to_sample = int(np.ceil(len(indices) * ratio))
            shuffled_indices = torch.randperm(len(indices), generator=generator).tolist()
            sampled_local_indices = shuffled_indices[:num_to_sample]
            subset_indices.extend([indices[i] for i in sampled_local_indices])

    random.shuffle(subset_indices)

    # Calculate and print the "after" counts for verification
    subset_labels = [full_dataset.labels[i] for i in subset_indices]
    if subset_labels:
        subset_counts = np.bincount(subset_labels)
        not_cancer_count = subset_counts[0] if len(subset_counts) > 0 else 0
        cancer_count = subset_counts[1] if len(subset_counts) > 1 else 0
    else:
        not_cancer_count, cancer_count = 0, 0

    print(f"    {split_name.title()} subset class counts -> CANCER: {cancer_count}, NOT_CANCER: {not_cancer_count}")

    return Subset(full_dataset, subset_indices)

In [ ]:
def clear_gpu():
    print("Clearing GPU cache...")
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as e:
            # Optional: only print if something really went wrong
            print(f"[clear_gpu] Warning: {e}")
    print("GPU cache cleared.")

In [ ]:
def load_and_select_models(metadata_dir, n_top_models, sort_metric):
    # --- Validate sort_metric ---
    valid_sort_metrics = ["best_validation_DICE", "best_val_auprc_score"]
    if sort_metric not in valid_sort_metrics:
        raise ValueError(f"Invalid sort_metric: '{sort_metric}'. Must be one of {valid_sort_metrics}")

    metadata_files = [os.path.join(metadata_dir, f) for f in os.listdir(metadata_dir) if f.endswith("_meta.json")]
    if not metadata_files:
        raise FileNotFoundError(f"No metadata files found in {metadata_dir}")

    all_metadata = []
    print(f"Loading metadata from {metadata_dir}...")
    for f_path in tqdm(metadata_files, desc="Loading Metadata"):
        try:
            with open(f_path, 'r') as f:
                meta = json.load(f)
                meta['metadata_filename'] = os.path.basename(f_path) # Add filename

                # --- Validate the sort metric value BEFORE adding ---
                metric_value = meta.get(sort_metric)
                if metric_value is not None and isinstance(metric_value, (int, float)) and not np.isnan(metric_value): # Check for NaN too
                    all_metadata.append(meta)
                else:
                    print(f"\nWarning: Skipping {os.path.basename(f_path)} - missing, non-numeric, or NaN sort metric '{sort_metric}' (value: {metric_value}).")
        except Exception as e:
            print(f"\nError loading metadata from {f_path}: {e}")

    if not all_metadata:
         raise ValueError("No valid metadata loaded after filtering for sort metric.")

    # --- Determine Sort Order based on the validated sort_metric ---
    if sort_metric == "best_validation_DICE":
        sort_descending = True # Maximize Optimized Dice
        # Default value for sorting if key is missing (shouldn't happen after filter)
        default_sort_value = -np.inf
    elif sort_metric == "best_val_auprc_score":
        sort_descending = True # Maximize Validation Dice
        default_sort_value = -np.inf
    # No else needed due to initial validation

    # --- Sort the list ---
    try:
        all_metadata.sort(
            key=lambda x: x.get(sort_metric, default_sort_value), # Use validated metric
            reverse=sort_descending
        )
        print(f"\nSorted {len(all_metadata)} models by '{sort_metric}' ({'Descending' if sort_descending else 'Ascending'}).")
    except Exception as e:
        print(f"Error during sorting: {e}")
        raise

    # --- Select Top N ---
    print(f"\n--- Selecting Top {n_top_models} Models based on {sort_metric} ---")
    top_models_meta = all_metadata[:n_top_models]
    if len(top_models_meta) < n_top_models:
        print(f"Warning: Only found {len(top_models_meta)} valid models after sorting, using all of them.")
    if not top_models_meta:
         raise ValueError("No models available after sorting and selection.")

    # Print selected models
    for i, meta in enumerate(top_models_meta):
        metric_val_display = meta.get(sort_metric, 'N/A')
        try: # Format as float if possible
            metric_display_str = f"{metric_val_display:.4f}"
        except:
            metric_display_str = str(metric_val_display) # Fallback to string

        print(f" {i+1}. Arch: {meta.get('architecture', 'N/A')}, Enc: {meta.get('encoder', 'N/A')}, "
              f"{sort_metric}: {metric_display_str}, " # Use formatted string
              f"Checkpoint: {os.path.basename(meta.get('checkpoint_path', 'N/A'))}")
    print("-" * 60) # Adjusted separator width

    return top_models_meta

In [ ]:
import contextlib

def predict_with_tta(model, images):
    """
    Run the model with simple test-time augmentations and average
    the cancer-channel probabilities.

    Args:
        model: segmentation model (may return (logits, aux))
        images: tensor [B, 3, H, W] already on the correct device

    Returns:
        probs_cancer: [B, H, W] averaged over TTA transforms
                      (foreground / cancer channel)
    """
    tta_probs = []

    def forward_pass(x, use_amp=False):
        ctx = (
            torch.amp.autocast("cuda")
            if (use_amp and x.is_cuda)
            else contextlib.nullcontext()
        )

        with torch.inference_mode(), ctx:
            out = model(x)

            if isinstance(out, (tuple, list)):
                out = out[0]

            # Binary (1-channel) vs multi-class (2-channel) outputs
            if out.shape[1] == 1:
                # [B, 1, H, W] → sigmoid → [B, H, W]
                probs = torch.sigmoid(out).squeeze(1)
            else:
                # [B, 2, H, W] → softmax → cancer channel
                probs = torch.softmax(out, dim=1)[:, 1, :, :]

            # Extra safeguard: remove NaN / Inf
            probs = torch.nan_to_num(
                probs,
                nan=0.0,
                posinf=1.0,
                neginf=0.0
            )

            probs = probs.clamp(0.0, 1.0)

        return probs

    # 1) Original
    probs = forward_pass(images)
    tta_probs.append(probs)

    # 2) Horizontal flip
    imgs_h = torch.flip(images, dims=[3])          # flip width
    probs_h = forward_pass(imgs_h)
    probs_h = torch.flip(probs_h, dims=[2])        # unflip prediction (width axis)
    tta_probs.append(probs_h)

    # 3) Vertical flip
    imgs_v = torch.flip(images, dims=[2])          # flip height
    probs_v = forward_pass(imgs_v)
    probs_v = torch.flip(probs_v, dims=[1])        # unflip prediction (height axis)
    tta_probs.append(probs_v)

    # Stack: [T, B, H, W] → mean over TTA transforms → [B, H, W]
    probs_cancer = torch.stack(tta_probs, dim=0).mean(dim=0)

    return probs_cancer

In [ ]:
def cache_predictions_fast_memmap(models_list, dataloader, device, cache_dir="pred_cache"):
    """
    Fast caching:
      - Creates one .npy per model via numpy.memmap and writes batches directly.
      - Saves ground truths once as a single .pt tensor.
      - Cleans NaN/Inf per-batch (so models are not excluded).

    Returns:
        cached_pred_paths, cached_trues_path, N_total, H, W, kept_model_indices
    """
    print(f"\n--- Caching predictions (FAST memmap) to: {cache_dir} ---")
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
    os.makedirs(cache_dir, exist_ok=True)

    for m in models_list:
        m.eval()

    # ---- Determine N_total and (H,W) from first valid batch ----
    print("Counting effective samples in dataloader (1 pass)...")
    N_total = count_effective_samples(dataloader)
    if N_total == 0:
        raise RuntimeError("No valid samples found in dataloader (after filtering).")

    H = W = None
    for batch in dataloader:
        if batch is None:
            continue
        images, masks = batch
        if images is None or masks is None:
            continue
        H, W = masks.shape[-2], masks.shape[-1]
        break

    if H is None or W is None:
        raise RuntimeError("Could not infer H,W from dataloader.")

    cached_pred_paths = []
    kept_model_indices = []  # <-- NEW
    cached_trues_path = os.path.join(cache_dir, "ground_truths.pt")

    trues_accum = []

    with torch.inference_mode():
        for model_idx, model in enumerate(models_list):
            print(f"\nCaching Preds (Model {model_idx+1}/{len(models_list)}) via memmap...")

            final_pred_path = os.path.join(cache_dir, f"model_{model_idx}_preds.npy")

            pred_mm = np.memmap(
                final_pred_path,
                dtype="float32",
                mode="w+",
                shape=(N_total, H, W),
            )

            write_pos = 0
            nan_count = 0
            total_count = 0

            for batch in tqdm(dataloader, desc=f"Model {model_idx+1}"):
                if batch is None:
                    continue
                images, masks = batch
                if images is None or masks is None:
                    continue

                if images.ndim == 3:
                    images = images.unsqueeze(0)
                if masks.ndim == 3:
                    masks = masks.unsqueeze(0)

                bsz = images.shape[0]
                images = images.to(device, non_blocking=True)

                if model_idx == 0:
                    trues_accum.append(masks[:, 1, :, :].cpu())

                probs_cancer = predict_with_tta(model, images)  # [B,H,W]

                probs_cancer = probs_cancer.detach().float().cpu()

                finite = torch.isfinite(probs_cancer)
                nan_count += int((~finite).sum().item())
                total_count += int(probs_cancer.numel())

                probs_cancer = torch.nan_to_num(
                    probs_cancer, nan=0.0, posinf=1.0, neginf=0.0
                ).clamp(0.0, 1.0)

                pred_mm[write_pos:write_pos+bsz, :, :] = probs_cancer.numpy()
                write_pos += bsz

            if write_pos != N_total:
                # clean partial file
                try:
                    del pred_mm
                    os.remove(final_pred_path)
                except:
                    pass
                raise RuntimeError(
                    f"Model {model_idx+1}: wrote {write_pos} samples but expected {N_total}. "
                    "Your dataloader filtering may be inconsistent between passes."
                )

            pred_mm.flush()
            del pred_mm

            nan_ratio = (nan_count / max(total_count, 1)) if total_count > 0 else 0.0
            if nan_ratio > 0:
                print(f"[WARN] Model {model_idx+1}: {nan_ratio:.4%} NaN/Inf pixels encountered (cleaned).")

            # probe without loading all
            probe = np.memmap(final_pred_path, dtype="float32", mode="r", shape=(N_total, H, W))
            maxv = float(probe.max())
            meanv = float(probe.mean())
            p99  = float(np.quantile(probe, 0.99))
            print(f"[DIAG] Model {model_idx+1}: max={maxv:.8e} mean={meanv:.8e} p99={p99:.8e}")

            if maxv < 1e-6:
                print(
                    f"[WARN] Model {model_idx+1}: near-zero outputs (max={maxv:.2E}). Excluding model."
                )
                # remove file and DO NOT keep the index
                try:
                    del probe
                    os.remove(final_pred_path)
                except:
                    pass
                continue

            # <-- KEEP
            cached_pred_paths.append(final_pred_path)
            kept_model_indices.append(model_idx)

    if len(trues_accum) == 0:
        raise RuntimeError("Failed to cache ground truths.")
    consolidated_trues = torch.cat(trues_accum, dim=0)  # [N_total,H,W]
    torch.save(consolidated_trues, cached_trues_path)
    del trues_accum, consolidated_trues
    gc.collect()

    if len(cached_pred_paths) == 0:
        raise RuntimeError("All models failed; no cached predictions created.")

    print(f"\nCaching complete. Using {len(cached_pred_paths)} valid models for ensemble.")
    return cached_pred_paths, cached_trues_path, N_total, H, W, kept_model_indices

In [ ]:
def compute_global_metrics_from_counts(tp: int, fp: int, fn: int, tn: int, smooth: float = 1e-6):
    """
    Compute micro Dice, IoU and MCC from global TP/FP/FN/TN counts.

    Returns:
        dice_micro (float), iou (float), mcc (float)
    """
    # Micro Dice over all pixels
    denom_dice = 2 * tp + fp + fn
    if denom_dice > 0:
        dice_micro = float((2.0 * tp + smooth) / (denom_dice + smooth))
    else:
        dice_micro = 0.0

    # IoU (Jaccard)
    denom_iou = tp + fp + fn
    if denom_iou > 0:
        iou = float((tp + smooth) / (denom_iou + smooth))
    else:
        iou = 0.0

    # Matthews Correlation Coefficient
    tp_fp = tp + fp
    tp_fn = tp + fn
    tn_fp = tn + fp
    tn_fn = tn + fn

    denom_mcc = (tp_fp * tp_fn * tn_fp * tn_fn) ** 0.5
    if denom_mcc > 0:
        mcc = float((tp * tn - fp * fn) / (denom_mcc + 1e-12))
    else:
        mcc = 0.0

    return dice_micro, iou, mcc

In [ ]:
def find_threshold_with_tpr_target_on_cached(
    ensemble_probs_cancer: torch.Tensor,
    true_masks: torch.Tensor,
    tpr_target: float = 0.95,
    secondary: str = "tnr",          # "tnr" or "mcc"
    num_steps: int = 100,
    smooth: float = 1e-6,
    chunk_size: int = 512,
    tie_break: str = "first",        # "first" keeps old implicit behavior
):
    """
    Policy:
      - Choose threshold with TPR >= tpr_target
      - Among those, maximize secondary (TNR or MCC)
      - If none meet target: maximize TPR, then secondary
    """
    assert secondary in ("tnr", "mcc")
    assert tie_break in ("first", "higher_thr", "lower_thr")

    # Ensure CPU tensors (recommended for large cached tensors)
    # (Safe even if already on CPU)
    if ensemble_probs_cancer.is_cuda:
        ensemble_probs_cancer = ensemble_probs_cancer.detach().cpu()
    if true_masks.is_cuda:
        true_masks = true_masks.detach().cpu()

    thresholds = torch.linspace(0.01, 0.99, num_steps)
    num_images = int(ensemble_probs_cancer.shape[0])

    # Use bool masks once
    true_masks_bool = true_masks.bool()

    def eval_threshold(thr: float):
        total_tp = total_fp = total_fn = total_tn = 0

        for start_idx in range(0, num_images, chunk_size):
            end_idx = min(start_idx + chunk_size, num_images)

            probs_chunk = ensemble_probs_cancer[start_idx:end_idx]   # [B,H,W]
            trues_chunk = true_masks_bool[start_idx:end_idx]         # [B,H,W] bool

            preds = probs_chunk >= thr                               # bool
            tp = (preds & trues_chunk).sum().item()
            fp = (preds & (~trues_chunk)).sum().item()
            fn = ((~preds) & trues_chunk).sum().item()
            tn = ((~preds) & (~trues_chunk)).sum().item()

            total_tp += tp; total_fp += fp; total_fn += fn; total_tn += tn

        tpr = total_tp / max((total_tp + total_fn), 1)
        tnr = total_tn / max((total_tn + total_fp), 1)

        if secondary == "tnr":
            sec_score = tnr
        else:
            _, _, mcc = compute_global_metrics_from_counts(
                total_tp, total_fp, total_fn, total_tn, smooth=smooth
            )
            sec_score = mcc

        stats = {"TPR": tpr, "TNR": tnr, "TP": total_tp, "FP": total_fp, "FN": total_fn, "TN": total_tn}
        return tpr, tnr, float(sec_score), stats

    # ---- Phase 1: enforce TPR constraint, maximize secondary ----
    best_thr = None
    best_secondary = -1e18
    best_stats = None

    for thresh in thresholds:
        thr = float(thresh.item())
        tpr, tnr, sec_score, stats = eval_threshold(thr)

        if tpr >= tpr_target:
            better = sec_score > best_secondary

            # Optional explicit tie-break (old behavior: keep first)
            if (not better) and abs(sec_score - best_secondary) < 1e-12 and best_thr is not None:
                if tie_break == "higher_thr":
                    better = thr > best_thr
                elif tie_break == "lower_thr":
                    better = thr < best_thr
                # tie_break == "first": better stays False (keeps first)

            if better or best_thr is None:
                best_secondary = sec_score
                best_thr = thr
                best_stats = stats

    # ---- Phase 2 fallback: maximize TPR, then secondary ----
    if best_thr is None:
        best_tpr = -1e18
        best_secondary = -1e18
        best_thr = 0.5
        best_stats = None

        for thresh in thresholds:
            thr = float(thresh.item())
            tpr, tnr, sec_score, stats = eval_threshold(thr)

            if (tpr > best_tpr) or (abs(tpr - best_tpr) < 1e-12 and sec_score > best_secondary):
                best_tpr = tpr
                best_secondary = sec_score
                best_thr = thr
                best_stats = stats

    return float(best_thr), float(best_secondary), best_stats

In [ ]:
def find_optimal_threshold_on_cached(
    ensemble_probs_cancer: torch.Tensor,
    true_masks: torch.Tensor,
    metric: str = 'mcc',
    num_steps: int = 100,
    smooth: float = 1e-6,
    chunk_size: int = 512,
):
    """
    Scan thresholds on pre-cached ensemble probabilities and choose the best one.

    Args:
        ensemble_probs_cancer: [N, H, W] float tensor with ensemble cancer probabilities.
        true_masks:           [N, H, W] int/bool tensor with {0,1} ground truth.
        metric:               'mcc', 'dice', or 'iou' — which metric to optimize.
        num_steps:            number of threshold points between 0.01 and 0.99.
        chunk_size:           how many images to process per chunk (for memory).

    Returns:
        best_threshold (float), best_score (float for the selected metric).
    """
    assert metric in ('mcc', 'dice', 'iou'), "metric must be 'mcc', 'dice', or 'iou'"

    thresholds = torch.linspace(0.01, 0.99, num_steps)
    num_images = ensemble_probs_cancer.shape[0]

    dice_scores = np.zeros(num_steps, dtype=np.float64)
    iou_scores  = np.zeros(num_steps, dtype=np.float64)
    mcc_scores  = np.zeros(num_steps, dtype=np.float64)

    print(f"\nFinding Optimal Threshold over {num_steps} steps...")
    pbar_thresh = tqdm(thresholds, desc="Finding Optimal Threshold", leave=True)

    true_masks = true_masks.int()

    for i, thresh in enumerate(pbar_thresh):
        threshold_value = float(thresh.item())

        total_tp = 0
        total_fp = 0
        total_fn = 0
        total_tn = 0

        # Process in chunks to avoid huge temporary tensors
        for start_idx in range(0, num_images, chunk_size):
            end_idx = min(start_idx + chunk_size, num_images)

            probs_chunk = ensemble_probs_cancer[start_idx:end_idx]  # [B,H,W]
            trues_chunk = true_masks[start_idx:end_idx]              # [B,H,W]

            # Binary predictions for this threshold
            preds_chunk = (probs_chunk >= threshold_value).int()

            tp = ((preds_chunk == 1) & (trues_chunk == 1)).sum().item()
            fp = ((preds_chunk == 1) & (trues_chunk == 0)).sum().item()
            fn = ((preds_chunk == 0) & (trues_chunk == 1)).sum().item()
            tn = ((preds_chunk == 0) & (trues_chunk == 0)).sum().item()

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_tn += tn

        dice_micro, iou, mcc = compute_global_metrics_from_counts(
            total_tp, total_fp, total_fn, total_tn, smooth=smooth
        )

        dice_scores[i] = dice_micro
        iou_scores[i]  = iou
        mcc_scores[i]  = mcc

        # Optional debug for a specific threshold (e.g. 0.5)
        if abs(threshold_value - 0.5) < 1e-6:
            print(
                f"[DEBUG Thresh {threshold_value:.2f}] "
                f"TP: {total_tp}, FN: {total_fn}, FP: {total_fp}, TN: {total_tn} | "
                f"Dice_micro: {dice_micro:.6f}, IoU: {iou:.6f}, MCC: {mcc:.6f}"
            )

    # Choose which array to optimize
    if metric == 'mcc':
        metric_scores = mcc_scores
    elif metric == 'dice':
        metric_scores = dice_scores
    else:  # 'iou'
        metric_scores = iou_scores

    # print(
    #     f"[DEBUG] Threshold search ({metric}, micro): min={float(metric_scores.min()):.6f}, "
    #     f"max={float(metric_scores.max()):.6f}"
    # )

    best_idx = int(np.argmax(metric_scores))
    best_threshold = float(thresholds[best_idx].item())
    best_metric_score = float(metric_scores[best_idx])

    # Also show the other metrics at this threshold for interpretability
    best_dice = float(dice_scores[best_idx])
    best_iou  = float(iou_scores[best_idx])
    best_mcc  = float(mcc_scores[best_idx])

    # print(
    #     f"[DEBUG] Best threshold = {best_threshold:.4f} | "
    #     f"Dice_micro={best_dice:.6f}, IoU={best_iou:.6f}, MCC={best_mcc:.6f} "
    #     f"(optimized for {metric})"
    # )

    return best_threshold, best_metric_score

In [ ]:
# ==============================================================================
# --- 13. Main Training Loop ---
# ==============================================================================
print(f"\n{'='*25} Starting Main Training Process {'='*25}")

fold_zip_filename = f'MASTER_SET_1.zip'
fold_zip_path = os.path.join(DATASET_ZIP_DIR, fold_zip_filename)

# --- Extract Dataset ---
print(f"Extracting Fold...")
if not os.path.exists(fold_zip_path):
  print(f"Zip not found: {fold_zip_path}. Skip.");
try:
    if os.path.exists(base_data_dir):
      shutil.rmtree(base_data_dir)
    os.makedirs(base_data_dir, exist_ok=True);

    with zipfile.ZipFile(fold_zip_path,'r') as z:
      z.extractall(base_data_dir)

    print("Extracted. Verifying...");

    if not os.path.isdir(train_cancer_image_dir) or not os.listdir(train_cancer_image_dir):
      raise RuntimeError("Verify failed")

    print("Verified.")

except Exception as e:
  print(f"Extract Err: {e}. Skip.");
  clear_gpu();

In [ ]:
# --- DataLoaders (with optional stratified subsampling) ---
print("\nCreating DataLoaders...")

sample_size = int(STATS_SAMPLE_SIZE) if STATS_SAMPLE_SIZE is not None else None
mean_to_be_used = MEAN if (MEAN is not None and len(MEAN) > 0) else None
std_to_be_used = STD if (STD is not None and len(STD) > 0) else None

try:
    # Always instantiate the full datasets first
    # 1) TRAIN: compute dataset-specific stats on full TRAIN
    full_train_ds = ProstateCancerDataset(
        train_cancer_image_dir,
        train_cancer_mask_dir,
        train_not_cancer_image_dir,
        train_not_cancer_mask_dir,
        compute_stats=False,         # <--- compute on full TRAIN
        stats_sample_size=sample_size,
        mean = mean_to_be_used,
        std = std_to_be_used
    )

    train_mean = full_train_ds.mean
    train_std  = full_train_ds.std

    # 2) VAL: reuse TRAIN stats (no compute_stats here!)
    full_val_ds = ProstateCancerDataset(
        val_cancer_image_dir,
        val_cancer_mask_dir,
        val_not_cancer_image_dir,
        val_not_cancer_mask_dir,
        mean=train_mean,
        std=train_std,
    )

    # --- NEW: Print "Before" counts ---
    val_counts_before = full_val_ds.get_class_counts()

    print("\n--- Full Dataset Class Counts (Before Subsampling) ---")
    print(f" VALIDATION: CANCER={val_counts_before['CANCER']}, NOT_CANCER={val_counts_before['NOT_CANCER']}")
    print('-'*50)

    def collate_fn(batch):
      batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
      return torch.utils.data.dataloader.default_collate(batch) if batch else None

    # --- Stratified Subsampling Logic ---
    if USE_SUBSET and SUBSET_RATIO < 1.0:
        print(f"Subsampling enabled. Using {SUBSET_RATIO:.0%} of the data for VALIDATION.")

        # --- MODIFICATION: Call the single, correct function for both datasets ---
        val_ds = create_stratified_subset(full_val_ds, SUBSET_RATIO, split_name="validation")

        print(f"\n--- Subset DS Lengths ---\n Val: {len(val_ds)}\n{'-'*30}")

    else:
        print("Using full datasets for training and validation.")
        val_ds = full_val_ds


    val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True, persistent_workers=WORKERS>0, prefetch_factor=2 if WORKERS>0 else None, collate_fn=collate_fn)


    # --- SANITY CHECK: Check for Data Leakage ---
    print("\n--- performing data leakage check ---")
    try:
        # Get a sample
        chk_img, chk_mask = val_ds[0]

        # chk_mask is (2, H, W). chk_img is (3, H, W).
        # Convert mask channel 1 (cancer) to comparable shape
        mask_c1 = chk_mask[1, :, :].numpy()

        # Convert image to grayscale for rough comparison
        img_gray = chk_img.mean(dim=0).numpy()

        # Check correlation or identity
        print(f"Sample Image Mean: {img_gray.mean():.4f}, Sample Mask Mean: {mask_c1.mean():.4f}")

        if np.allclose(img_gray, mask_c1, atol=1e-1):
            print("CRITICAL WARNING: Image and Mask appear statistically identical. Check your directory paths!")
            print(f"Img Dir: {val_cancer_image_dir}")
            print(f"Mask Dir: {val_cancer_mask_dir}")
    except Exception as e:
        print(f"Could not perform leakage check: {e}")

    print("---------------------------------------\n")
    print("DataLoaders created successfully.")
except Exception as e:
  print(f"DataLoader Err: {e}. Skip.")
  clear_gpu()

In [ ]:
def change_paths(top_models_metadata):
  for meta in top_models_metadata:
      if not os.path.exists(meta['checkpoint_path']):
          new_path = str(meta['checkpoint_path']).replace(WHERE_WAS_CREATED, CURRENT_ENV)
          if os.path.exists(new_path):
              meta['checkpoint_path'] = new_path
              print(f"Updated path for {meta['checkpoint_path']}")
          else:
              print(f"Path not found: {new_path}")
  return top_models_metadata

In [ ]:
def load_checkpoint_strict_without_aux(model, chkpt_path, device):
    # 1) Load checkpoint safely on CPU
    chkpt = torch.load(chkpt_path, map_location="cpu")

    # 2) Extract model state dict
    state_dict = chkpt.get("model_state_dict", chkpt)

    # 3) Drop auxiliary / classification head keys if present
    state_dict = {
        k: v for k, v in state_dict.items()
        if not k.startswith("classification_head.")
    }

    # 4) Load STRICTLY while model is still on CPU
    missing, unexpected = model.load_state_dict(state_dict, strict=True)

    if unexpected:
        print(f"[WARN] Unexpected keys: {unexpected[:5]}{'...' if len(unexpected) > 5 else ''}")
    if missing:
        print(f"[WARN] Missing keys: {missing[:5]}{'...' if len(missing) > 5 else ''}")

    # 5) Move model to device (single allocation)
    model.to(device)

    # 6) Re-compile ONLY if original model was compiled
    if chkpt.get("is_compiled", False):
        print("[INFO] Re-compiling model (checkpoint was compiled)")
        try:
            model = torch.compile(model)
        except Exception as e:
            print(f"[WARN] torch.compile failed: {e}")

    return model

In [ ]:
print('Loading and Selecting Top N Models...')
top_models_metadata = load_and_select_models(METADATA_DIR, N_TOP_MODELS, SORT_METRIC)

print("Changing paths...")
top_models_metadata = change_paths(top_models_metadata)

ensemble_models = []

constituent_model_info = []
for meta in top_models_metadata:
    # (Same model loading logic as your script)
    try:
        arch, enc, chkpt_path = meta['architecture'], meta['encoder'], meta['checkpoint_path']
        model = get_model(architecture=arch, encoder=enc, validation=True)

        model = load_checkpoint_strict_without_aux(model, chkpt_path, device)

        model.eval()
        ensemble_models.append(model)
        constituent_model_info.append({k: meta.get(k) for k in ['architecture',
                                                                'encoder',
                                                                'checkpoint_path',
                                                                'optimal_threshold_optimized',
                                                                'best_val_auprc_score',
                                                                'best_model_epoch',
                                                                'Learning_rate',
                                                                'Encoder_LR_Factor',
                                                                'Weight_Decay',
                                                                'Batch_Size',
                                                                'Num_Epochs',
                                                                'Workers',
                                                                'Seed',
                                                                'DATASET_ZIP_DIR',
                                                                'Dropout',
                                                                'patience',
                                                                'Loss',
                                                                'Optimizer'
                                                                ]})
    except Exception as e:
        print(f"Error loading model {meta.get('checkpoint_path')}: {e}")

if len(ensemble_models) < 2:
    print("Error: Fewer than 2 models loaded. Exiting.")
    exit(1)

if len(ensemble_models)<int(N_TOP_MODELS):
  N_EFFECTIVE_MODELS = len(ensemble_models)
  print(f"Warning: Only {len(ensemble_models)} models loaded. Using them all.")
else:
  N_EFFECTIVE_MODELS = int(N_TOP_MODELS)
  print(f"Using the top {N_EFFECTIVE_MODELS} models.")

In [ ]:
cached_pred_paths, cached_trues_path, N_total_cached, H_cached, W_cached, kept_model_indices = cache_predictions_fast_memmap(
    ensemble_models,
    val_loader,
    device,
    cache_dir=PRED_CACHE_DIR
)

# Load ground truths ONCE (CPU tensor)
cached_trues_tensor = load_cached_ground_truths(cached_trues_path)

# Create numpy.memmap objects for each model's prediction file
model_pred_memmaps = [
    np.memmap(path, dtype="float32", mode="r", shape=(N_total_cached, H_cached, W_cached))
    for path in cached_pred_paths
]

N_EFFECTIVE_MODELS = len(model_pred_memmaps)

In [ ]:
def make_stratified_train_holdout_split_from_trues(trues, holdout_frac, seed):
    """
    Splits indices into (opt_pool_idx, holdout_idx) with stratification by has_tumor.
    trues: torch.Tensor [N,H,W] 0/1
    """
    N = int(trues.shape[0])
    rng = np.random.default_rng(int(seed))

    has_tumor = (trues.sum(dim=(1,2)) > 0).cpu().numpy()
    pos_idx = np.where(has_tumor)[0]
    neg_idx = np.where(~has_tumor)[0]

    rng.shuffle(pos_idx)
    rng.shuffle(neg_idx)

    n_holdout_pos = int(round(len(pos_idx) * holdout_frac))
    n_holdout_neg = int(round(len(neg_idx) * holdout_frac))

    holdout_idx = np.concatenate([pos_idx[:n_holdout_pos], neg_idx[:n_holdout_neg]])
    opt_pool_idx = np.concatenate([pos_idx[n_holdout_pos:], neg_idx[n_holdout_neg:]])

    rng.shuffle(holdout_idx)
    rng.shuffle(opt_pool_idx)

    return opt_pool_idx.astype(np.int64), holdout_idx.astype(np.int64)

def make_stratified_subset_indices_from_trues_with_pool(trues, pool_idx, subset_size, seed, pos_frac=0.5):
    """
    Choose a stratified subset only from pool_idx.
    """
    subset_size = min(int(subset_size), int(len(pool_idx)))
    rng = np.random.default_rng(int(seed))

    has_tumor_full = (trues.sum(dim=(1,2)) > 0).cpu().numpy()
    pool_has_tumor = has_tumor_full[pool_idx]

    pos_pool = pool_idx[np.where(pool_has_tumor)[0]]
    neg_pool = pool_idx[np.where(~pool_has_tumor)[0]]

    rng.shuffle(pos_pool)
    rng.shuffle(neg_pool)

    n_pos = min(int(round(subset_size * pos_frac)), len(pos_pool))
    n_neg = min(subset_size - n_pos, len(neg_pool))

    chosen = np.concatenate([pos_pool[:n_pos], neg_pool[:n_neg]])
    rng.shuffle(chosen)
    return chosen.astype(np.int64)

def report_subset_stats(trues, idx, name="subset"):
    has_tumor = (trues.sum(dim=(1,2)) > 0)
    full_pos = int(has_tumor.sum().item())
    full_n   = int(trues.shape[0])

    sub_has_tumor = has_tumor[idx]
    sub_pos = int(sub_has_tumor.sum().item())
    sub_n   = int(len(idx))

    print(f"[{name}] size={sub_n}/{full_n} | pos={sub_pos} ({sub_pos/max(sub_n,1):.2%}) "
          f"| full_pos={full_pos} ({full_pos/max(full_n,1):.2%})")

# --- 1) Create OPT pool and HOLDOUT split from full validation ---
opt_pool_idx, holdout_idx = make_stratified_train_holdout_split_from_trues(
    cached_trues_tensor,
    holdout_frac=VAL_HOLDOUT_FRAC,
    seed=VAL_HOLDOUT_SEED
)

report_subset_stats(cached_trues_tensor, opt_pool_idx, name="opt_pool")
report_subset_stats(cached_trues_tensor, holdout_idx, name="holdout")

# --- 2) Build Optuna subset only from OPT pool (never from holdout) ---
if USE_OPTUNA_SUBSET:
    opt_idx = make_stratified_subset_indices_from_trues_with_pool(
        trues=cached_trues_tensor,
        pool_idx=opt_pool_idx,
        subset_size=OPTUNA_SUBSET_SIZE,
        seed=OPTUNA_SUBSET_SEED,
        pos_frac=OPTUNA_SUBSET_POS_FRAC
    )
    cached_trues_subset = cached_trues_tensor[opt_idx]
    report_subset_stats(cached_trues_tensor, opt_idx, name="optuna_subset")
else:
    opt_idx = opt_pool_idx
    cached_trues_subset = cached_trues_tensor[opt_idx]
    report_subset_stats(cached_trues_tensor, opt_idx, name="optuna_full_opt_pool")

# Holdout tensor for final reporting
cached_trues_holdout = cached_trues_tensor[holdout_idx]

In [ ]:
# Free up VRAM by deleting models after caching predictions
del ensemble_models
clear_gpu()
if torch.cuda.is_available():
    torch.cuda.synchronize()

In [ ]:
def eval_fixed_threshold_on_cached(ensemble_probs, true_masks, thr, chunk_size=512):
    if ensemble_probs.is_cuda: ensemble_probs = ensemble_probs.cpu()
    if true_masks.is_cuda: true_masks = true_masks.cpu()

    true_bool = true_masks.bool()
    N = int(ensemble_probs.shape[0])

    total_tp = total_fp = total_fn = total_tn = 0
    for s in range(0, N, chunk_size):
        e = min(s + chunk_size, N)
        probs = ensemble_probs[s:e]
        trues = true_bool[s:e]
        preds = probs >= float(thr)

        total_tp += (preds & trues).sum().item()
        total_fp += (preds & (~trues)).sum().item()
        total_fn += ((~preds) & trues).sum().item()
        total_tn += ((~preds) & (~trues)).sum().item()

    tpr = total_tp / max(total_tp + total_fn, 1)
    tnr = total_tn / max(total_tn + total_fp, 1)
    dice_micro, iou, mcc = compute_global_metrics_from_counts(total_tp, total_fp, total_fn, total_tn)

    return {
        "thr": float(thr),
        "TP": total_tp, "FP": total_fp, "FN": total_fn, "TN": total_tn,
        "TPR": float(tpr), "TNR": float(tnr),
        "dice_micro": float(dice_micro),
        "iou": float(iou),
        "mcc": float(mcc),
    }

In [ ]:
def build_ensemble_probs_chunked(
    weights_normalized,
    model_pred_memmaps,
    N_total,
    H,
    W,
    chunk_size: int = 128,
):
    """
    Build weighted ensemble probabilities on CPU using chunking.

    Assumptions:
      - Each memmap already contains finite values in [0,1] (you did nan_to_num + clamp in caching).
      - weights_normalized sums to 1.

    Returns:
      ensemble_probs: torch.FloatTensor [N_total, H, W] on CPU in [0,1]
    """
    w = np.asarray(weights_normalized, dtype=np.float32)
    assert len(w) == len(model_pred_memmaps)

    out = torch.empty((N_total, H, W), dtype=torch.float32)  # CPU output

    for s in range(0, N_total, chunk_size):
        e = min(s + chunk_size, N_total)

        # accumulate in torch on CPU
        acc = None
        for i, mm in enumerate(model_pred_memmaps):
            # mm[s:e] returns a numpy view; from_numpy is a zero-copy CPU tensor view
            x = torch.from_numpy(mm[s:e])  # [b,H,W], float32 CPU

            if acc is None:
                acc = x * w[i]
            else:
                acc.add_(x, alpha=float(w[i]))  # acc += x * w[i]

        # store chunk
        out[s:e].copy_(acc)

        # free chunk accumulator (important on Colab)
        del acc

    return out.clamp_(0.0, 1.0)

In [ ]:
def build_ensemble_probs_chunked_subset(
    weights_normalized,
    model_pred_memmaps,
    indices,          # np.ndarray [K]
    H,
    W,
    chunk_size=256,   # chunks over K, not N
):
    """
    Build ensemble probabilities only for a subset of samples.

    model_pred_memmaps: list of np.memmap, each shaped [N_total, H, W] float32
    indices: subset indices into N_total
    returns: torch.FloatTensor [K, H, W] on CPU
    """
    w = np.asarray(weights_normalized, dtype=np.float32)
    K = int(len(indices))
    out = torch.empty((K, H, W), dtype=torch.float32)

    for s in range(0, K, chunk_size):
        e = min(s + chunk_size, K)
        idx = indices[s:e]

        acc = None
        for i, mm in enumerate(model_pred_memmaps):
            # mm[idx] returns a real numpy array [b,H,W] (advanced indexing)
            x = torch.from_numpy(mm[idx])
            if acc is None:
                acc = x * float(w[i])
            else:
                acc.add_(x, alpha=float(w[i]))
        out[s:e].copy_(acc)
        del acc

    return out.clamp(0.0, 1.0)

In [ ]:
def objective(trial):
    # 1) Suggest raw weights in [0,1]
    weights_raw = [trial.suggest_float(f"w_{i}", 0.0, 1.0) for i in range(N_EFFECTIVE_MODELS)]

    # 2) Normalize weights
    sum_weights = float(sum(weights_raw))
    if sum_weights == 0.0:
        weights_normalized = [1.0 / N_EFFECTIVE_MODELS] * N_EFFECTIVE_MODELS
    else:
        weights_normalized = [float(w) / sum_weights for w in weights_raw]

    # 3) Build ensemble probs (subset if enabled)
    if USE_OPTUNA_SUBSET:
        ensemble_probs = build_ensemble_probs_chunked_subset(
            weights_normalized=weights_normalized,
            model_pred_memmaps=model_pred_memmaps,
            indices=opt_idx,
            H=H_cached,
            W=W_cached,
            chunk_size=OPTUNA_ENSEMBLE_CHUNK,
        )
        trues = cached_trues_subset
    else:
        # fallback to full val (slow, but correct)
        full_idx = np.arange(N_total_cached, dtype=np.int64)
        ensemble_probs = build_ensemble_probs_chunked_subset(
            weights_normalized=weights_normalized,
            model_pred_memmaps=model_pred_memmaps,
            indices=full_idx,
            H=H_cached,
            W=W_cached,
            chunk_size=OPTUNA_ENSEMBLE_CHUNK,
        )
        trues = cached_trues_tensor

    # 4) Sanity check
    if float(ensemble_probs.max()) < 1e-6:
        del ensemble_probs
        gc.collect()
        raise optuna.exceptions.TrialPruned()

    # 5) Find best threshold for these weights (KEEP YOUR POLICY)
    if THRESHOLD_POLICY == "tpr_target":
        opt_threshold, opt_score, stats = find_threshold_with_tpr_target_on_cached(
            ensemble_probs_cancer=ensemble_probs,
            true_masks=trues,
            tpr_target=TPR_TARGET,
            secondary=SECONDARY_METRIC,  # "tnr" or "mcc"
            num_steps=THRESH_NUM_STEPS_TRIALS,
            chunk_size=THRESH_SCAN_CHUNK,
        )
        trial.set_user_attr("threshold_stats", stats)
    else:
        opt_threshold, opt_score = find_optimal_threshold_on_cached(
            ensemble_probs_cancer=ensemble_probs,
            true_masks=trues,
            metric=METRIC_TO_OPTIMIZE,
            num_steps=THRESH_NUM_STEPS_TRIALS,
            chunk_size=THRESH_SCAN_CHUNK,
        )

    # 6) Store useful info
    trial.set_user_attr("optimal_threshold", float(opt_threshold))
    trial.set_user_attr("normalized_weights", weights_normalized)

    del ensemble_probs
    gc.collect()

    return float(opt_score)

In [ ]:
# ### REFACTORED: Step 3 - Run the Optuna optimization study
print(f"\n--- Starting Optuna Optimization ({N_OPTUNA_TRIALS} trials) ---")

# We want to maximize the metric (e.g., Dice score)
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED) # TPESampler is the default, good for this task
)

# Start the optimization
study.optimize(objective, n_trials=N_OPTUNA_TRIALS)

# --- Post-Optimization Analysis ---
print("\n--- Optimization Complete ---")
print(f"Number of finished trials: {len(study.trials)}")

best_trial = study.best_trial


if THRESHOLD_POLICY == "tpr_target":
    OPTUNA_OBJECTIVE_NAME = f"{SECONDARY_METRIC}_at_tpr{TPR_TARGET:.2f}"
else:
    OPTUNA_OBJECTIVE_NAME = METRIC_TO_OPTIMIZE

print(f"Best trial value ({OPTUNA_OBJECTIVE_NAME}): {best_trial.value:.6f}")

best_weights = best_trial.user_attrs['normalized_weights']


# ============================
# Threshold selection on OPT POOL (NOT holdout)
# ============================
ensemble_probs_optpool = build_ensemble_probs_chunked_subset(
    weights_normalized=best_weights,
    model_pred_memmaps=model_pred_memmaps,
    indices=opt_pool_idx,
    H=H_cached,
    W=W_cached,
    chunk_size=OPTUNA_ENSEMBLE_CHUNK,
)

if THRESHOLD_POLICY == "tpr_target":
    thr_opt, score_opt, stats_opt = find_threshold_with_tpr_target_on_cached(
        ensemble_probs_cancer=ensemble_probs_optpool,
        true_masks=cached_trues_tensor[opt_pool_idx],
        tpr_target=TPR_TARGET,
        secondary=SECONDARY_METRIC,
        num_steps=THRESH_NUM_STEPS_FINAL,
        chunk_size=THRESH_SCAN_CHUNK,
    )
else:
    thr_opt, score_opt = find_optimal_threshold_on_cached(
        ensemble_probs_cancer=ensemble_probs_optpool,
        true_masks=cached_trues_tensor[opt_pool_idx],
        metric=METRIC_TO_OPTIMIZE,
        num_steps=THRESH_NUM_STEPS_FINAL,
        chunk_size=THRESH_SCAN_CHUNK,
    )

print(f"[OPT POOL] Selected threshold = {thr_opt:.4f}")

# ============================
# FINAL EVALUATION ON HOLDOUT (NO TUNING)
# ============================
ensemble_probs_holdout = build_ensemble_probs_chunked_subset(
    weights_normalized=best_weights,
    model_pred_memmaps=model_pred_memmaps,
    indices=holdout_idx,
    H=H_cached,
    W=W_cached,
    chunk_size=OPTUNA_ENSEMBLE_CHUNK,
)

holdout_stats = eval_fixed_threshold_on_cached(
    ensemble_probs_holdout,
    cached_trues_holdout,
    thr=thr_opt,
    chunk_size=THRESH_SCAN_CHUNK,
)

print(
    f"[HOLDOUT] thr(from opt_pool)={thr_opt:.4f} | "
    f"TPR={holdout_stats['TPR']:.4f} "
    f"TNR={holdout_stats['TNR']:.4f} "
    f"MCC={holdout_stats['mcc']:.6f} "
    f"Dice_micro={holdout_stats['dice_micro']:.6f}"
)

In [ ]:
# Save detailed results of all trials to a CSV
timestamp = get_formatted_datetime_string()
csv_results=f'CSV_results_{timestamp}.csv'

In [ ]:
# Save detailed results of all trials to a CSV
results_df = study.trials_dataframe()
csv_path = os.path.join(OUTPUT_DIR, csv_results)
results_df.to_csv(csv_path, index=False)
print(f"\nFull optimization results saved to: {csv_path}")

In [ ]:
best_weights = best_trial.user_attrs['normalized_weights']

effective_constituent_model_info = [constituent_model_info[i] for i in kept_model_indices]

assert len(best_weights) == len(effective_constituent_model_info), "Weight/metadata length mismatch"

constituent_models_with_weights = []
print("Best trial parameters (weights per model):")
for weight, model_info in zip(best_weights, effective_constituent_model_info):
    model_info_with_weight = model_info.copy()
    model_info_with_weight["ensemble_weight"] = float(weight)
    constituent_models_with_weights.append(model_info_with_weight)

    arch = model_info.get("architecture", "N/A")
    enc  = model_info.get("encoder", "N/A")
    print(f"  - Weight: {float(weight):.4f} -> Model: {arch} ({enc})")

In [ ]:
# Decide objective name once (this is what Optuna maximized per trial)
if THRESHOLD_POLICY == "tpr_target":
    OPTUNA_OBJECTIVE_NAME = f"{SECONDARY_METRIC}_at_tpr{TPR_TARGET:.2f}"
else:
    OPTUNA_OBJECTIVE_NAME = METRIC_TO_OPTIMIZE

best_ensemble_meta = {
    # What Optuna optimized during trials (on the Optuna subset)
    "optuna_best_trial_value": float(best_trial.value),
    "optuna_objective_name": str(OPTUNA_OBJECTIVE_NAME),

    # Threshold selection outcome (chosen ONLY on opt-pool)
    "threshold_selected_on": "opt_pool",
    "opt_pool_threshold": float(thr_opt),
    "opt_pool_score": float(score_opt),
}

# Only attach stats_opt if you actually computed it (tpr_target path)
# stats_opt comes from find_threshold_with_tpr_target_on_cached(...)
if THRESHOLD_POLICY == "tpr_target" and stats_opt is not None:
    best_ensemble_meta["opt_pool_threshold_stats"] = stats_opt

# Final unbiased evaluation (fixed thr_opt, evaluated on holdout)
best_ensemble_meta["holdout"] = {
    "fraction": float(VAL_HOLDOUT_FRAC),
    "seed": int(VAL_HOLDOUT_SEED),
    "stats": holdout_stats,  # dict from eval_fixed_threshold_on_cached(...)
}

# Ensemble composition
best_ensemble_meta["constituent_models"] = constituent_models_with_weights

# Experiment bookkeeping
best_ensemble_meta["optimization_details"] = {
    "num_trials": int(N_OPTUNA_TRIALS),
    "optuna_sampler_seed": int(SEED),

    "use_subset": bool(USE_SUBSET),
    "subset_ratio": float(SUBSET_RATIO),
    "validation_total_size": int(len(val_ds)),

    "use_optuna_subset": bool(USE_OPTUNA_SUBSET),
    "optuna_subset_size": int(len(opt_idx)) if opt_idx is not None else None,
    "optuna_subset_seed": int(OPTUNA_SUBSET_SEED),

    # if you’re stratifying the optuna subset
    "optuna_subset_pos_frac": 0.5,

    "threshold_policy": str(THRESHOLD_POLICY),
    "tpr_target": float(TPR_TARGET) if THRESHOLD_POLICY == "tpr_target" else None,
    "secondary_metric": str(SECONDARY_METRIC) if THRESHOLD_POLICY == "tpr_target" else None,
    "metric_optimized": str(METRIC_TO_OPTIMIZE) if THRESHOLD_POLICY != "tpr_target" else None,

    "threshold_steps_trials": int(THRESH_NUM_STEPS_TRIALS),
    "threshold_steps_final": int(THRESH_NUM_STEPS_FINAL),

    "datetime": str(timestamp),
}

In [ ]:
meta_path_ensemble = "ENSEMBLE_OPTIMIZATION_"+timestamp+".json"
os.makedirs(OUTPUT_DIR, exist_ok=True)
meta_path = os.path.join(OUTPUT_DIR, meta_path_ensemble)
with open(meta_path, 'w') as f:
    json.dump(best_ensemble_meta, f, indent=4)
print(f"Best ensemble metadata saved to: {meta_path}")

In [ ]:
# --- Cleanup ---
print("\\nCleaning up cached predictions and extracted data...")

# Remove the on-disk cache
cache_dir = PRED_CACHE_DIR
if os.path.exists(cache_dir):
    try:
        shutil.rmtree(cache_dir)
        print("Cleaned prediction cache directory.")
    except Exception as e:
        print(f"Cache cleanup err: {e}")

print("\nCleaning up extracted validation data...")
if os.path.exists(base_data_dir):
  try:
    shutil.rmtree(base_data_dir)
    print("Cleaned data dir.")
  except Exception as e:
    print(f"Data cleanup err: {e}")

print("\n--- Ensemble Weight Optimization Script Finished ---")